# Loading and Preprocessing

In [1]:
from data_loader import load_data

In [2]:
X_clinical,\
X_clinical_A,\
X_clinical_B,\
X_clinical_Delta,\
targets = load_data()

In [3]:
targets.head()

,surv_bestresponse_car,ae_summ_icans_v2,ae_summ_icans_highestgrade_v2,ae_summ_crs_v2,ae_summ_highestgrade_v2
record_id,,,,,
FTC-EMC-0022,1.0,0.0,0.0,0.0,0.0
FTC-EMC-0023,2.0,1.0,4.0,1.0,2.0
FTC-EMC-0027,4.0,1.0,1.0,1.0,2.0
FTC-UMCG-0001,1.0,1.0,1.0,1.0,1.0
FTC-UMCG-0003,4.0,0.0,0.0,1.0,1.0


In [4]:
datasets = {
    "Clinical": X_clinical,
    "Clinical_A": X_clinical_A,
    "Clinical_B": X_clinical_B,
    "Clinical_Delta": X_clinical_Delta
}

# Feature Selection
Dropping highly correlated clinical and radiomics features using the EDA analysis, as they add little to no new information.

## Clinical highly correlated features  
We're using the EDA because it took the mixed feature types into account when calculating the correlations, and there are only a few features, so we could manually write them. So, it was more straight-forward than redoing it ourselves.

In [5]:
clinical_ftrs_to_drop = ['total_num_priortherapylines_fl',
                'tr_car_bridg_type.factor_None',
                'scr_weight',
                'scr_height',
                'indication_age_60',
                'cli_st_neutrophils',
                'indication_extran_invol',
                'indication_pri_refr',
                'indication_sec_refr'
                ]

In [6]:
for name, df in datasets.items():
    print(f"Before dropping features in {name}: {df.shape}")
    df.drop(columns=clinical_ftrs_to_drop, inplace=True)
    print(f"After dropping features in {name}: {df.shape}")

Before dropping features in Clinical: (85, 35)
After dropping features in Clinical: (85, 26)
Before dropping features in Clinical_A: (85, 134)
After dropping features in Clinical_A: (85, 125)
Before dropping features in Clinical_B: (85, 134)
After dropping features in Clinical_B: (85, 125)
Before dropping features in Clinical_Delta: (85, 134)
After dropping features in Clinical_Delta: (85, 125)


## Highly Correlated Radiomic Features
These are all numerical values, with high dimensionality. The EDA already showed a large number of highly correlated features, so we will write code to automatically keep only one of possibly multiple highly correlated features.

In [7]:
import numpy as np

def keep_uncorrelated_features(df, threshold=0.9):
    # Absolute correlation matrix
    corr = df.corr().abs()
    # Upper triangle (ignore diagonal and lower half), k=1 means we start from the first diagonal above the main diagonal
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    # Find columns with any correlation above threshold
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any() and upper[col].dtype in [np.float64]]
    # Return reduced dataframe
    return df.drop(columns=to_drop)

# Usage
for name, df in datasets.items():
    print(f"Before dropping correlated features in {name}: {df.shape}")

    datasets[name] = keep_uncorrelated_features(df, threshold=0.9)
    print(f"After dropping correlated features in {name}: {datasets[name].shape}")

Before dropping correlated features in Clinical: (85, 26)
After dropping correlated features in Clinical: (85, 26)
Before dropping correlated features in Clinical_A: (85, 125)
After dropping correlated features in Clinical_A: (85, 62)
Before dropping correlated features in Clinical_B: (85, 125)
After dropping correlated features in Clinical_B: (85, 59)
Before dropping correlated features in Clinical_Delta: (85, 125)
After dropping correlated features in Clinical_Delta: (85, 71)


In [8]:
cat_cols = datasets['Clinical'].select_dtypes(exclude='number').columns.tolist()

In [9]:
# Using ColumnTransformer to apply different transformations to numeric and categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Create a ColumnTransformer to apply StandardScaler to numeric columns and leave categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 'passthrough', cat_cols)],
        remainder= StandardScaler()
    
)



In [10]:
from model_eval import result_viewer

# ICANS 0,1

In [11]:
y = targets['ae_summ_icans_v2']

In [12]:
y.value_counts()

ae_summ_icans_v2
1.0    48
0.0    37
Name: count, dtype: int64

In [13]:
clinical_ftrs = datasets['Clinical'].columns.tolist()
radio_b = datasets['Clinical_B'].drop(clinical_ftrs, axis=1)

In [14]:
process_B = StandardScaler()

In [15]:
icans_1 = result_viewer(process_B, {'only_radio_B': radio_b}, y)

Results for Logistic Regression:


,only_radio_B
test_roc_auc,0.5527 ± 0.1351
train_roc_auc,0.9042 ± 0.0283
test_precision,0.6144 ± 0.1146
train_precision,0.8317 ± 0.0406
test_recall,0.6044 ± 0.0726
train_recall,0.8960 ± 0.0282
test_f1,0.6002 ± 0.0490
train_f1,0.8625 ± 0.0344


Results for Decision Tree:


,only_radio_B
test_roc_auc,0.4737 ± 0.1019
train_roc_auc,0.7876 ± 0.0515
test_precision,0.5070 ± 0.0882
train_precision,0.7645 ± 0.1145
test_recall,0.7156 ± 0.3178
train_recall,0.8590 ± 0.1588
test_f1,0.5683 ± 0.2082
train_f1,0.7864 ± 0.0426


Results for Random Forest:


,only_radio_B
test_roc_auc,0.5504 ± 0.1268
train_roc_auc,0.9838 ± 0.0106
test_precision,0.5788 ± 0.0385
train_precision,0.8368 ± 0.0369
test_recall,0.8378 ± 0.1736
train_recall,1.0000 ± 0.0000
test_f1,0.6766 ± 0.0798
train_f1,0.9107 ± 0.0225


In [16]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6590 ± 0.1154,0.5370 ± 0.1011,0.5984 ± 0.1271,0.4618 ± 0.1066
train_roc_auc,0.8798 ± 0.0277,0.9880 ± 0.0075,0.9908 ± 0.0063,0.9982 ± 0.0019
test_precision,0.7265 ± 0.1513,0.6238 ± 0.1339,0.6827 ± 0.1701,0.5885 ± 0.0526
train_precision,0.8059 ± 0.0261,0.9690 ± 0.0101,0.9397 ± 0.0251,0.9845 ± 0.0127
test_recall,0.7333 ± 0.0961,0.6911 ± 0.1615,0.6644 ± 0.1141,0.6889 ± 0.1063
train_recall,0.8646 ± 0.0303,0.9740 ± 0.0166,0.9688 ± 0.0194,0.9742 ± 0.0229
test_f1,0.7183 ± 0.0804,0.6466 ± 0.1094,0.6454 ± 0.0340,0.6313 ± 0.0671
train_f1,0.8341 ± 0.0262,0.9714 ± 0.0100,0.9540 ± 0.0206,0.9791 ± 0.0104


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6121 ± 0.1364,0.4835 ± 0.1178,0.6215 ± 0.1313,0.4608 ± 0.1428
train_roc_auc,0.7936 ± 0.0474,0.8125 ± 0.0387,0.8373 ± 0.0228,0.8360 ± 0.0229
test_precision,0.6137 ± 0.0826,0.5733 ± 0.0949,0.5802 ± 0.0568,0.5398 ± 0.1904
train_precision,0.7259 ± 0.0747,0.7469 ± 0.0685,0.7795 ± 0.0781,0.8516 ± 0.0825
test_recall,0.7978 ± 0.1399,0.6511 ± 0.1097,0.8356 ± 0.1372,0.5467 ± 0.2817
train_recall,0.9323 ± 0.0424,0.9221 ± 0.0492,0.9065 ± 0.0740,0.8081 ± 0.1765
test_f1,0.6863 ± 0.0769,0.6049 ± 0.0846,0.6805 ± 0.0721,0.5190 ± 0.1899
train_f1,0.8123 ± 0.0361,0.8212 ± 0.0271,0.8317 ± 0.0231,0.8078 ± 0.0797


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6244 ± 0.1082,0.3924 ± 0.0785,0.6063 ± 0.1115,0.5050 ± 0.0910
train_roc_auc,0.9560 ± 0.0123,0.9678 ± 0.0108,0.9912 ± 0.0038,0.9901 ± 0.0067
test_precision,0.6133 ± 0.0835,0.5791 ± 0.0725,0.5702 ± 0.0485,0.5535 ± 0.0614
train_precision,0.7983 ± 0.0520,0.8131 ± 0.0522,0.8606 ± 0.0536,0.8612 ± 0.0360
test_recall,0.8178 ± 0.1152,0.7489 ± 0.1137,0.8133 ± 0.1529,0.7578 ± 0.1606
train_recall,0.9634 ± 0.0267,0.9686 ± 0.0307,0.9947 ± 0.0105,0.9842 ± 0.0211
test_f1,0.6917 ± 0.0488,0.6414 ± 0.0229,0.6666 ± 0.0785,0.6291 ± 0.0562
train_f1,0.8719 ± 0.0327,0.8827 ± 0.0306,0.9217 ± 0.0280,0.9180 ± 0.0210


## ICANS >=2

In [17]:
y = targets["ae_summ_icans_highestgrade_v2"]

In [18]:
y = y.astype('int8')

In [19]:
y[y<2] = 0
y[y>=2] = 1


In [20]:
y.value_counts()

ae_summ_icans_highestgrade_v2
0    52
1    33
Name: count, dtype: int64

In [21]:
icans_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6478 ± 0.1157,0.5673 ± 0.1479,0.6312 ± 0.1346,0.6275 ± 0.1652
train_roc_auc,0.8942 ± 0.0167,0.9873 ± 0.0121,0.9975 ± 0.0024,0.9924 ± 0.0072
test_precision,0.5676 ± 0.1111,0.5142 ± 0.1608,0.5333 ± 0.1752,0.5717 ± 0.1825
train_precision,0.7998 ± 0.0302,0.9226 ± 0.0349,0.9775 ± 0.0298,0.9683 ± 0.0291
test_recall,0.5000 ± 0.2352,0.6190 ± 0.2228,0.4571 ± 0.2121,0.5571 ± 0.2282
train_recall,0.6353 ± 0.0710,0.9014 ± 0.0452,0.9319 ± 0.0149,0.9251 ± 0.0522
test_f1,0.4877 ± 0.1377,0.5415 ± 0.1499,0.4660 ± 0.1532,0.5382 ± 0.1778
train_f1,0.7062 ± 0.0501,0.9117 ± 0.0379,0.9539 ± 0.0183,0.9458 ± 0.0392


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4986 ± 0.1255,0.4776 ± 0.0490,0.5235 ± 0.0782,0.6047 ± 0.1467
train_roc_auc,0.7541 ± 0.0357,0.7874 ± 0.0547,0.7990 ± 0.0530,0.8067 ± 0.0361
test_precision,0.3924 ± 0.1140,0.3790 ± 0.0388,0.3389 ± 0.1928,0.4702 ± 0.2073
train_precision,0.7825 ± 0.1802,0.7609 ± 0.1277,0.8132 ± 0.1530,0.8074 ± 0.1393
test_recall,0.3476 ± 0.3305,0.3048 ± 0.0933,0.3095 ± 0.2274,0.4714 ± 0.2504
train_recall,0.5909 ± 0.2458,0.6422 ± 0.1254,0.6803 ± 0.1568,0.7088 ± 0.1998
test_f1,0.3161 ± 0.1569,0.3327 ± 0.0766,0.3117 ± 0.1918,0.4619 ± 0.2121
train_f1,0.6092 ± 0.1309,0.6798 ± 0.0627,0.7086 ± 0.0407,0.7161 ± 0.0623


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5553 ± 0.1125,0.4998 ± 0.2347,0.5435 ± 0.1247,0.3931 ± 0.0730
train_roc_auc,0.9586 ± 0.0070,0.9861 ± 0.0051,0.9929 ± 0.0039,0.9940 ± 0.0052
test_precision,0.2667 ± 0.3887,0.3667 ± 0.3712,0.4667 ± 0.3232,0.4000 ± 0.4899
train_precision,0.9778 ± 0.0444,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_recall,0.0667 ± 0.0816,0.0952 ± 0.0782,0.1571 ± 0.1061,0.0619 ± 0.0762
train_recall,0.3932 ± 0.0917,0.4325 ± 0.0954,0.5607 ± 0.0441,0.5678 ± 0.0631
test_f1,0.1016 ± 0.1260,0.1460 ± 0.1215,0.2333 ± 0.1587,0.1071 ± 0.1317
train_f1,0.5554 ± 0.0987,0.5981 ± 0.0867,0.7175 ± 0.0365,0.7223 ± 0.0505


# CRS 0,1

In [22]:
y = targets["ae_summ_crs_v2"]

In [23]:
y.value_counts()

ae_summ_crs_v2
1.0    79
0.0     6
Name: count, dtype: int64

In [24]:
crs_1 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4525 ± 0.1685,0.2008 ± 0.2007,0.3992 ± 0.0844,0.4275 ± 0.1689
train_roc_auc,0.9861 ± 0.0071,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_precision,0.9256 ± 0.0255,0.9272 ± 0.0225,0.9236 ± 0.0333,0.9272 ± 0.0225
train_precision,0.9434 ± 0.0105,0.9938 ± 0.0077,0.9784 ± 0.0073,0.9875 ± 0.0062
test_recall,0.9492 ± 0.0470,0.9625 ± 0.0306,0.8675 ± 0.2350,0.9625 ± 0.0306
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9367 ± 0.0297,0.9439 ± 0.0129,0.8784 ± 0.1669,0.9439 ± 0.0129
train_f1,0.9708 ± 0.0056,0.9969 ± 0.0039,0.9891 ± 0.0037,0.9937 ± 0.0031


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4567 ± 0.1541,0.4750 ± 0.0234,0.3367 ± 0.2724,0.4183 ± 0.0576
train_roc_auc,0.8997 ± 0.0381,0.7922 ± 0.0943,0.9411 ± 0.0445,0.7573 ± 0.0917
test_precision,0.9242 ± 0.0246,0.9256 ± 0.0217,0.9379 ± 0.0049,0.9224 ± 0.0239
train_precision,0.9574 ± 0.0145,0.9695 ± 0.0133,0.9844 ± 0.0001,0.9547 ± 0.0093
test_recall,0.9242 ± 0.0246,0.9375 ± 0.0395,0.9608 ± 0.0529,0.8992 ± 0.0494
train_recall,0.9905 ± 0.0077,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9238 ± 0.0168,0.9306 ± 0.0137,0.9486 ± 0.0289,0.9096 ± 0.0264
train_f1,0.9736 ± 0.0062,0.9845 ± 0.0069,0.9922 ± 0.0000,0.9768 ± 0.0049


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.3783 ± 0.1007,0.3067 ± 0.1733,0.6250 ± 0.2469,0.1792 ± 0.1059
train_roc_auc,1.0000 ± 0.0000,0.9981 ± 0.0038,0.9994 ± 0.0013,0.9987 ± 0.0016
test_precision,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235
train_precision,0.9294 ± 0.0059,0.9349 ± 0.0069,0.9322 ± 0.0115,0.9378 ± 0.0107
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129,0.9633 ± 0.0129
train_f1,0.9634 ± 0.0031,0.9664 ± 0.0037,0.9649 ± 0.0061,0.9678 ± 0.0057


# CRS >=2

In [25]:
y = targets["ae_summ_highestgrade_v2"]
y = y.astype('int8')
y[y<2] = 0
y[y>=2] = 1

y.value_counts()

ae_summ_highestgrade_v2
0    55
1    30
Name: count, dtype: int64

In [26]:
crs_2 = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5909 ± 0.1063,0.6485 ± 0.1664,0.5333 ± 0.1558,0.5697 ± 0.2165
train_roc_auc,0.8735 ± 0.0260,0.9894 ± 0.0078,0.9930 ± 0.0047,0.9996 ± 0.0008
test_precision,0.5083 ± 0.2892,0.5929 ± 0.1555,0.4667 ± 0.3293,0.4937 ± 0.2797
train_precision,0.7070 ± 0.0730,0.9482 ± 0.0176,0.9635 ± 0.0342,0.9913 ± 0.0174
test_recall,0.2667 ± 0.1333,0.4000 ± 0.2494,0.2667 ± 0.1700,0.4000 ± 0.2261
train_recall,0.5500 ± 0.1034,0.9167 ± 0.0264,0.8917 ± 0.0773,0.9417 ± 0.0333
test_f1,0.3117 ± 0.1067,0.4572 ± 0.2079,0.3263 ± 0.1984,0.3980 ± 0.1677
train_f1,0.6165 ± 0.0873,0.9321 ± 0.0211,0.9250 ± 0.0522,0.9656 ± 0.0218


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.3985 ± 0.0731,0.5727 ± 0.1191,0.4758 ± 0.0828,0.6364 ± 0.1317
train_roc_auc,0.7706 ± 0.0450,0.8146 ± 0.0364,0.8009 ± 0.0239,0.8028 ± 0.0390
test_precision,0.2650 ± 0.1367,0.3589 ± 0.2126,0.3867 ± 0.3357,0.2780 ± 0.2281
train_precision,0.7064 ± 0.0911,0.8510 ± 0.1392,0.9188 ± 0.0752,0.7539 ± 0.1714
test_recall,0.2000 ± 0.0667,0.3333 ± 0.2357,0.1667 ± 0.1054,0.4667 ± 0.4522
train_recall,0.5417 ± 0.1646,0.6000 ± 0.2261,0.5417 ± 0.0950,0.6500 ± 0.2965
test_f1,0.2149 ± 0.0691,0.3229 ± 0.2024,0.2107 ± 0.1220,0.3263 ± 0.2780
train_f1,0.5867 ± 0.0961,0.6545 ± 0.1330,0.6719 ± 0.0670,0.6207 ± 0.1052


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5667 ± 0.1330,0.6455 ± 0.1052,0.6121 ± 0.1553,0.6364 ± 0.1460
train_roc_auc,0.9530 ± 0.0103,0.9551 ± 0.0103,0.9742 ± 0.0162,0.9953 ± 0.0049
test_precision,0.0000 ± 0.0000,0.1667 ± 0.2108,0.2000 ± 0.4000,0.4000 ± 0.4899
train_precision,1.0000 ± 0.0000,0.9413 ± 0.0584,1.0000 ± 0.0000,1.0000 ± 0.0000
test_recall,0.0000 ± 0.0000,0.0667 ± 0.0816,0.0333 ± 0.0667,0.0667 ± 0.0816
train_recall,0.2417 ± 0.0890,0.5583 ± 0.0972,0.5167 ± 0.1253,0.5667 ± 0.0624
test_f1,0.0000 ± 0.0000,0.0944 ± 0.1160,0.0571 ± 0.1143,0.1143 ± 0.1400
train_f1,0.3810 ± 0.1155,0.6929 ± 0.0683,0.6716 ± 0.1184,0.7213 ± 0.0525


# Outcome
CMR vs others

In [27]:
y = targets["surv_bestresponse_car"]

In [28]:
y.value_counts()

surv_bestresponse_car
1.0    63
2.0    12
4.0     8
0.0     2
Name: count, dtype: int64

Early Death: 0, CMR: 1, PMR:2, SD: 3, PD:4

In [29]:
y = y.astype('int8')

In [30]:
y[y != 1] = 0

In [31]:
y.value_counts()

surv_bestresponse_car
1    63
0    22
Name: count, dtype: int64

In [32]:
outcome = result_viewer(preprocessor, datasets, y)

Results for Logistic Regression:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7105 ± 0.0623,0.5946 ± 0.1474,0.7054 ± 0.0881,0.6733 ± 0.0513
train_roc_auc,0.9076 ± 0.0186,0.9955 ± 0.0038,0.9973 ± 0.0029,0.9973 ± 0.0025
test_precision,0.7931 ± 0.0533,0.7934 ± 0.0995,0.8467 ± 0.0657,0.7716 ± 0.0572
train_precision,0.8555 ± 0.0295,0.9441 ± 0.0299,0.9692 ± 0.0196,0.9696 ± 0.0249
test_recall,0.8551 ± 0.0808,0.7949 ± 0.0579,0.7974 ± 0.1333,0.7936 ± 0.0789
train_recall,0.9325 ± 0.0159,0.9920 ± 0.0098,0.9920 ± 0.0098,0.9920 ± 0.0160
test_f1,0.8221 ± 0.0615,0.7888 ± 0.0512,0.8103 ± 0.0517,0.7803 ± 0.0532
train_f1,0.8920 ± 0.0163,0.9673 ± 0.0196,0.9804 ± 0.0139,0.9804 ± 0.0138


Results for Decision Tree:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6704 ± 0.1056,0.5163 ± 0.0612,0.6447 ± 0.1100,0.6113 ± 0.0718
train_roc_auc,0.7848 ± 0.0115,0.8143 ± 0.0517,0.7886 ± 0.0470,0.8472 ± 0.0276
test_precision,0.8129 ± 0.0594,0.7588 ± 0.0336,0.8075 ± 0.0756,0.7750 ± 0.0549
train_precision,0.8671 ± 0.0047,0.8634 ± 0.0302,0.8714 ± 0.0290,0.8872 ± 0.0098
test_recall,0.9205 ± 0.0718,0.9038 ± 0.0914,0.7936 ± 0.1252,0.8282 ± 0.1567
train_recall,0.9323 ± 0.0372,0.9605 ± 0.0411,0.9365 ± 0.0197,0.9642 ± 0.0151
test_f1,0.8589 ± 0.0153,0.8238 ± 0.0542,0.7955 ± 0.0835,0.7871 ± 0.0665
train_f1,0.8982 ± 0.0182,0.9080 ± 0.0047,0.9025 ± 0.0205,0.9239 ± 0.0017


Results for Random Forest:


,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.7577 ± 0.0510,0.5400 ± 0.1663,0.6321 ± 0.1376,0.6349 ± 0.0682
train_roc_auc,0.9684 ± 0.0079,0.9662 ± 0.0082,0.9817 ± 0.0090,0.9955 ± 0.0054
test_precision,0.7500 ± 0.0228,0.7412 ± 0.0288,0.7645 ± 0.0544,0.7349 ± 0.0263
train_precision,0.7829 ± 0.0130,0.8058 ± 0.0255,0.8239 ± 0.0207,0.8132 ± 0.0124
test_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,0.9538 ± 0.0615,0.9692 ± 0.0615
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.8569 ± 0.0151,0.8510 ± 0.0191,0.8450 ± 0.0157,0.8348 ± 0.0301
train_f1,0.8781 ± 0.0082,0.8923 ± 0.0156,0.9033 ± 0.0125,0.8969 ± 0.0075
